In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v3 as iio
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

from detect import detect_frame
from detect_utils import track_frames
from detect_utils import estimate_minmass
from detection_ui import setup_detection_ui

In [2]:
# Step 1: File browser with native file dialog
import tkinter as tk
from tkinter import filedialog

# Global state
stack = None
stack_shape = None

# Create root window (hidden)
root = tk.Tk()
root.withdraw()

# Let user browse for TIFF file
file_path = filedialog.askopenfilename(
    title='Select TIFF file',
    filetypes=[('TIFF files', '*.tif *.tiff'), ('All files', '*.*')],
    initialdir='.'
)

root.destroy()

if file_path:
    try:
        stack = iio.imread(file_path)
        stack_shape = stack.shape
        
        # Calculate image statistics
        img_min = float(stack.min())
        img_max = float(stack.max())
        
        print(f'✓ Loaded: {file_path}')
        print(f'  Shape: {stack_shape}')
        print(f'  Range: [{img_min:.0f}, {img_max:.0f}]')
    except Exception as e:
        print(f'Error loading file: {str(e)}')
        stack = None
        stack_shape = None
else:
    print('No file selected. Please run this cell again to browse for a file.')

✓ Loaded: C:/Users/andyr/Downloads/2022_09_14_Oridot_Cal_No_Focus_20ms_20p.tif
  Shape: (988, 684, 856)
  Range: [303, 38723]


In [ ]:
# Set up and display detection UI
results = setup_detection_ui(stack, stack_shape)

HTML(value='<hr>')

HTML(value='<b>Detection Output:</b>')

Output()

HTML(value='<b>Tracking Output:</b>')

Output()

In [ ]:
# Link and visualize particle trajectories
from trajectory_ui import setup_trajectory_ui

results = setup_trajectory_ui(results, stack)

HTML(value='<hr>')

HTML(value='<b>Trajectory Output:</b>')

Output()

{'tracks':         frame           x           y          mass       signal
 0           0  119.958531    3.656398  53281.756520  4734.753245
 1           0  319.019370    4.084746  26072.707870  3282.762250
 2           0  146.781690    5.262324  35857.864577  2903.981990
 3           0  222.694064    4.805936  27650.958952  3345.892293
 4           0  271.379845    5.472868  24431.326745  2714.591861
 ...       ...         ...         ...           ...          ...
 775527    900  219.605428  648.505219  24396.423148  2648.463473
 775528    900  295.460128  651.137959  63868.715298  7079.546592
 775529    900  279.103478  665.744348  58571.788352  6009.974805
 775530    900   81.961883  667.022422  22715.667482  2546.599494
 775531    900  102.615385  672.256410  25822.518865  2190.075564
 
 [775532 rows x 5 columns],
 'track_params': {'diameter': 9.0,
  'separation': 3.0,
  'minmass': 20000.0,
  'start_frame': 0,
  'end_frame': 900},
 'last_detection': None,
 'detect_params': None}

In [6]:
# Plot individual particle trajectory
particle_id = 0  # Change this to analyze different particles

if 'linked_trajectories' in results:
    traj = results['linked_trajectories'][results['linked_trajectories']['particle'] == particle_id].sort_values('frame')
    
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f'Particle {particle_id}: X vs Time', f'Particle {particle_id}: Y vs Time'))
    fig.add_trace(go.Scatter(x=traj['frame'], y=traj['x'], mode='lines+markers', name='X'), row=1, col=1)
    fig.add_trace(go.Scatter(x=traj['frame'], y=traj['y'], mode='lines+markers', name='Y'), row=1, col=2)
    fig.update_xaxes(title_text="Frame")
    fig.update_yaxes(title_text="X (pixels)", row=1, col=1)
    fig.update_yaxes(title_text="Y (pixels)", row=1, col=2)
    fig.update_layout(height=400, width=1000, showlegend=False)
    fig.show()
else:
    print("Run trajectory linking first!")

NameError: name 'make_subplots' is not defined